# UAV Swarm Coordination -- 3D Environment (all eight approaches)

This notebook runs the **3D** decentralized, SOC-aware UAV swarm environment
((x, y, z) positions with an altitude/climb-power energy model) with **eight
policies**:

| Policy | Type | Description |
|---|---|---|
| **Random** | Baseline | Uniformly random valid action every step (sanity floor). |
| **Greedy-Nearest** | Baseline | Decentralized, *not* SOC-aware: always heads for the nearest active task and ignores battery until it is already critical. |
| **CBBA** | Baseline | Consensus-Based Bundle Algorithm: SOC-aware bidding that factors in 3D energy cost (including climb cost) and refuses unsafe tasks, over the same lossy/stale channel. |
| **MAPPO** | Learned (CTDE) | Multi-Agent PPO (NumPy reference implementation) **ported to 3D**, behavior-cloned from Greedy-Nearest and fine-tuned with PPO + training-only proximity shaping. |
| **MADDPG** | Learned (CTDE) | Discrete-action MADDPG with a centralized critic / decentralized softmax actors, warm-started from the same demo buffer. |
| **QMIX** | Learned (CTDE) | Value-decomposition QMIX with a global-state mixing network; decentralized execution via per-agent utility argmax. |
| **DMPC** | Model-based | Distributed Model-Predictive Control over the known 3D dynamics (efficiency ceiling). |
| **HRL-H** | Proposed model | Hierarchical RL + heuristic navigation: a high-level learned allocator (MAPPO) with a low-level model-based executor (commit-and-fly, battery safety, deconfliction). |

Four environments are evaluated (saved as a **2x2** snapshot that can be re-plotted later):

| Environment id | Vertical layout | No-fly zones |
|---|---|---|
| **open** | Open volume (tasks scattered through altitude) | none |
| **layered** | Layered altitude bands | none |
| **open_nfz** | Open volume | 4 cuboid no-fly zones |
| **layered_nfz** | Layered altitude bands | 4 cuboid no-fly zones |

Training of MAPPO / MADDPG / QMIX is unchanged (one checkpoint, layered, obstacle-free).
Every trained and baseline policy is then evaluated in **all four** environments.

## What this notebook does

1. Configure the 3D environment (stress or easy preset) and the four evaluation environments.
2. Train the 3D MAPPO policy (BC warm-start + PPO fine-tune; variable-task capable).
3. Train the MADDPG and QMIX baselines from the same offline demo buffer.
4. Evaluate **all eight approaches** in each of the four environments on coverage, stranded
   UAVs, total reward, final SOC, episode length, plus extended operational metrics.
5. Run the stress sweeps -- **packet loss**, **comm range**, **task density**, **UAV count
   (scalability)**, **UAV dropout (fault tolerance)** -- **per environment**, and save
   JSON + CSV so figures and tables can be rebuilt later.
6. Save a **2x2 environment plot** (plus per-environment snapshot JSON), stress comparison,
   extended-metric graphs, latency, scenario comparison, and HRL-H ablations -- 4 panels
   use 2x2, 5-6 panels use 3x2, with large-size labels and informative legends.
7. Write comparison tables (CSV / JSON / Markdown) for all models x all environments.


In [ ]:
import sys, os, json, time, pickle
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# --- locate the bundled project code -----------------------------------
NB_DIR = os.path.abspath(os.environ.get("UAV_NB_DIR", os.getcwd()))
# `code` provides the marl_algorithms / model_based / hrl_h packages. The
# package dirs themselves are NOT added to sys.path because their flat module
# names (mlp, hrl_h, ...) would shadow the packages.
for _p in ["code", "code/env_3d", "code/mappo_3d"]:
    _p = os.path.join(NB_DIR, _p)
    if _p not in sys.path:
        sys.path.insert(0, _p)

from uav_swarm_env_3d import UAVSwarmEnv3D, EnvConfig3D
from baselines_3d import random_policy, greedy_nearest_policy, cbba_policy
from model import Actor, Critic, softmax
import train_3d as mt      # 3D MAPPO training helpers (numpy)

# --- new baselines + proposed model (3D) --------------------------------
from marl_algorithms.mlp import normalize_padded_obs
from marl_algorithms.maddpg import MADDPG, ReplayBuffer, update_from_buffer, \
    bc_warmstart_actor
from marl_algorithms.qmix import QMIX, build_state, qmix_update, \
    qmix_bc_warmstart
from model_based.dmpc import dmpc_policy
from hrl_h.hrl_h import HRLH

# --- output locations ----------------------------------------------------
FIG_DPI = 600               # publication-quality figures
OUT_DIR = os.path.join(NB_DIR, "outputs")
FIG_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

from visualize_3d import (
    plot_env_3d, plot_environments_2x2, collect_env_snapshot,
    save_env_snapshot, load_env_snapshot, EnvSnapshot, plot_env_from_snapshot,
    replot_environments_2x2_from_dir,
)
from eval_io import (
    apply_pub_style, panel_shape, figsize_for, write_json, write_csv,
    dump_sweep, dump_base, dump_sweep_by_env, dump_base_by_env, dump_tables,
    hide_unused_axes, FONT_TITLE, FONT_LABEL, FONT_TICK, FONT_LEGEND,
    FONT_SUPTITLE, FONT_ANNOT, SUMMARY_FIELDS, TABLE_METRICS,
)

apply_pub_style()
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["figure.dpi"] = 100

# --- consistent style for the eight approaches, used by every figure -----
POLICY_ORDER = ["random", "greedy", "cbba", "mappo", "maddpg", "qmix",
                "dmpc", "hrlh"]
POLICY_LABELS = {"random": "Random", "greedy": "Greedy-Nearest",
                 "cbba": "CBBA (SOC-aware)", "mappo": "MAPPO",
                 "maddpg": "MADDPG", "qmix": "QMIX", "dmpc": "DMPC",
                 "hrlh": "HRL-H (proposed)"}
POLICY_COLORS = {"random": "#d62728", "greedy": "#1f77b4",
                 "cbba": "#2ca02c", "mappo": "#9467bd",
                 "maddpg": "#ff7f0e", "qmix": "#17becf",
                 "dmpc": "#7f7f7f", "hrlh": "#e377c2"}
POLICY_MARKERS = {"random": "o", "greedy": "s", "cbba": "^", "mappo": "D",
                  "maddpg": "*", "qmix": "P", "dmpc": "x", "hrlh": "h"}

print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)
print("notebook dir:", NB_DIR)


## 1. Environment configuration

Two presets (same stress/easy split as the 2D project):

* **Stress config (default)** -- 8 agents / 8 tasks, 400 x 400 m footprint, 5-60 m altitude,
  12 m 3D comm range, 18 Wh battery, 0.25 safe-SOC floor. This is where SOC and communication
  constraints actually bind (matches the paper's main stress-test configuration).
* **Default/easy config** -- 6 agents / 12 tasks, 100 m area, 25 m comm range, 90 Wh battery.

Four **evaluation environments** are built from that preset:

* **open** -- open volume, no no-fly zones
* **layered** -- discrete altitude bands, no no-fly zones
* **open_nfz** -- open volume with 4 cuboid no-fly zones
* **layered_nfz** -- layered bands with 4 cuboid no-fly zones

`ENV_CFG` is the training config (layered, obstacle-free) so checkpoints stay compatible.
`ENV_CFGS` is the dict used by every evaluation sweep. Snapshot JSON is written under
`outputs/env_snapshots/` so the 2x2 environment figure can be re-plotted later.

`N_SEEDS` controls evaluation seeds per sweep level, `N_TRAIN_EPISODES` the PPO fine-tune
budget after the **behavior-cloning warm-start** (`WARM_START`), `SHAPING_BETA` the weight of
a **training-only** proximity shaping bonus, and `TRAIN_TASK_COUNTS` the task counts MAPPO
is trained on so it generalizes across the task-density sweep.


In [ ]:
USE_STRESS_CFG = True             # False -> easier default config
SCENARIO = "layered"              # training scenario (checkpoints stay layered)
N_SEEDS = 10                      # evaluation seeds per sweep level
WARM_START = True                 # BC warm-start from Greedy-Nearest before PPO fine-tune
BC_EPISODES = 400                 # greedy demonstration episodes for behavior cloning
BC_ITERS = 4000                   # behavior-cloning optimizer steps
ACTOR_HIDDEN = 64                 # actor hidden dim (capacity for the warm-started policy)
N_TRAIN_EPISODES = 600            # PPO fine-tune budget (after the BC warm-start)
PPO_EPISODES_PER_UPDATE = 40
TRAIN_TASK_COUNTS = [8, 16, 32]   # task counts sampled during MAPPO training
TARGET_PACKET_LOSS = 0.3          # packet-loss curriculum target
CURRICULUM_EPISODES = 300         # curriculum ramp length (episodes)
SEED = 0

NFZ_N_OBSTACLES = 4
NFZ_OBSTACLE_HALF = 20.0

if USE_STRESS_CFG:
    _base_kw = dict(
        n_agents=8, n_tasks=8, area_size=400.0, altitude_min=5.0, altitude_max=60.0,
        n_layers=3, comm_range=12.0, battery_capacity_wh=18.0, soc_min_safe=0.25,
        max_steps=500,
    )
    CFG_NAME = "stress"
else:
    _base_kw = dict(n_agents=6, n_tasks=12, max_steps=200)
    CFG_NAME = "default"

ENV_CFG = EnvConfig3D(**_base_kw, scenario=SCENARIO)

# Four evaluation environments: open / layered, each with and without no-fly zones.
ENV_SPECS = [
    ("open",        "Open volume",                  "volume",  False, 0),
    ("layered",     "Layered altitude bands",       "layered", False, 0),
    ("open_nfz",    "Open volume + no-fly zones",   "volume",  True,  NFZ_N_OBSTACLES),
    ("layered_nfz", "Layered + no-fly zones",       "layered", True,  NFZ_N_OBSTACLES),
]
ENV_ORDER = [s[0] for s in ENV_SPECS]
ENV_LABELS = {s[0]: s[1] for s in ENV_SPECS}
ENV_CFGS = {}
for _eid, _lab, _sc, _obs, _nobs in ENV_SPECS:
    ENV_CFGS[_eid] = EnvConfig3D(
        **_base_kw, scenario=_sc,
        obstacles_enabled=_obs, n_obstacles=_nobs,
        obstacle_half=NFZ_OBSTACLE_HALF,
    )

# Training-only reward shaping: a dense proximity bonus that rewards agents
# for being near active tasks (3D distance). It is applied ONLY to the PPO
# training signal; every reported metric (eval) uses the raw environment
# reward, so the eight approaches are still compared on the exact same reward.
SHAPING_BETA = 0.2
SHAPING_DMAX = float((ENV_CFG.area_size ** 2 * 3) ** 0.5)

# ---- off-policy MARL baselines (MADDPG / QMIX) -------------------------
MARL_DEMO_EPISODES = 120     # greedy (eps-explored) demos for the BC warm-start
MARL_TASK_COUNTS = [4, 8, 16]
MARL_EPS_DEMO = 0.1          # eps-greedy exploration during demo collection
MARL_BC_ITERS = 4000         # BC optimizer steps (actor / qnet)
MARL_RL_ITERS = 2000         # replay-buffer RL updates after the BC warm-start
MADDPG_LR = 3e-4
QMIX_LR = 1e-4
QMIX_BC_WEIGHT = 5.0         # BC regularization during QMIX TD updates
QMIX_Q_LIM = 50.0            # bootstrap-target clamp (offline-Q stability)
QMIX_TARGET_EVERY = 500      # hard target-network update interval

print("configuration:", CFG_NAME, "| train scenario:", SCENARIO)
print("evaluation environments:", ENV_ORDER)
for _eid in ENV_ORDER:
    _c = ENV_CFGS[_eid]
    print(f"  {_eid:<12} scenario={_c.scenario:<8} nfz={_c.n_obstacles}  {ENV_LABELS[_eid]}")
print(ENV_CFG)


## 2. Policies and evaluation helpers

`run_episode` returns a metric dict that covers both the *core* metrics shared with the 3D
notebook (so results are comparable across environments): total reward, tasks completed,
coverage fraction, number stranded, mean final SOC, SOC spread, episode length -- and the
*extended* operational metrics used in this notebook: energy efficiency (Wh per task),
total / per-task 3D distance, minimum SOC reached, fraction of agent-steps below the safe
floor, task-assignment contention, mean communication information-age (staleness), and
per-task completion latency. `run_episode` also supports `fail_frac` (agent-failure
injection) for the fault-tolerance sweep in section 7.5.

In [ ]:
def run_episode(policy_fn, env_cfg, seed, fail_frac=0.0):
    """Run one episode of `policy_fn` in a fresh 3D env; return a rich metric dict.

    Records the core metrics (reward, coverage, stranded, SOC, steps) plus the
    extended set used across the notebook: energy efficiency, minimum SOC / time
    below the safe floor, 3D distance travelled, task-assignment contention, mean
    communication information-age (staleness), and per-task completion latency.

    If `fail_frac > 0`, that fraction of agents is silently removed mid-episode
    (fault injection at a random step), so the returned metrics measure how
    robustly the policy keeps working after losing UAVs.
    """
    cfg = EnvConfig3D(**{**env_cfg.__dict__, "seed": seed})
    env = UAVSwarmEnv3D(cfg)
    env.reset()

    inject_step = None
    fail_idx = None
    if fail_frac > 0:
        # Probe a no-fault mission length so the injection lands mid-mission
        # (a random step in the 20%-70% window of the mission) instead of
        # after the episode has already ended.
        probe_cfg = EnvConfig3D(**{**env_cfg.__dict__, "seed": seed + 100000})
        probe_env = UAVSwarmEnv3D(probe_cfg)
        probe_env.reset()
        for _t in range(probe_cfg.max_steps):
            probe_actions = policy_fn(probe_env)
            _, _, _pt, _ptr, _ = probe_env.step(probe_actions)
            if all(_pt.values()) or all(_ptr.values()):
                break
        mission_len = _t + 1
        lo, hi = max(1, int(mission_len * 0.2)), max(2, int(mission_len * 0.7))
        inject_step = int(env.rng.integers(lo, hi))
        n_fail = max(1, int(round(cfg.n_agents * fail_frac)))
        fail_idx = env.rng.choice(cfg.n_agents, size=n_fail, replace=False)

    total_reward = 0.0
    steps = 0
    soc_log, pos_log, staleness_log = [], [], []
    contended_steps = 0
    completion_steps = []
    task_prev = env.task_completed.copy()
    for t in range(cfg.max_steps):
        if inject_step is not None and t == inject_step:
            env.stranded[fail_idx] = True
        actions = policy_fn(env)

        # contention: >=2 non-stranded agents targeting the same active task
        act = np.array([actions[a] for a in env.agents], dtype=int)
        engaged = act[(~env.stranded) & (act < env.n_tasks)]
        if len(engaged) > 1:
            engaged = engaged[env.task_active[engaged]]
            if len(np.unique(engaged)) < len(engaged):
                contended_steps += 1

        _, rewards, term, trunc, _ = env.step(actions)
        total_reward += sum(rewards.values())
        steps = t + 1
        soc_log.append(env.soc.copy())
        pos_log.append(env.pos.copy())
        staleness_log.append(
            float(env.staleness[~np.eye(env.n, dtype=bool)].mean()))

        newly_done = env.task_completed & ~task_prev
        if newly_done.any():
            for k in np.where(newly_done)[0]:
                completion_steps.append(steps)
        task_prev = env.task_completed.copy()
        if all(term.values()) or all(trunc.values()):
            break

    completed = int(env.task_completed.sum())
    expired = int(env.task_expired.sum())
    alive = ~env.stranded
    soc_arr = np.array(soc_log) if soc_log else np.zeros((1, env.n))
    pos_arr = np.array(pos_log) if pos_log else np.zeros((1, env.n, 3))
    dist_total = float(np.linalg.norm(np.diff(pos_arr, axis=0), axis=-1).sum())
    energy_wh = float((cfg.battery_capacity_wh * (1.0 - env.soc)).sum())
    return {
        "total_reward": float(total_reward),
        "tasks_completed": completed,
        "tasks_total": cfg.n_tasks,
        "tasks_expired": expired,
        "coverage_frac": completed / cfg.n_tasks,
        "expired_frac": expired / cfg.n_tasks,
        "n_stranded": int(env.stranded.sum()),
        "stranded_frac": float(env.stranded.mean()),
        "mean_final_soc": float(env.soc[alive].mean()) if alive.any() else 0.0,
        "soc_std": float(env.soc.std()),
        "steps_taken": steps,
        "energy_wh_total": energy_wh,
        "energy_wh_per_task": energy_wh / max(completed, 1),
        "dist_total_m": dist_total,
        "dist_per_task_m": dist_total / max(completed, 1),
        "min_soc": float(soc_arr.min()),
        "below_floor_pct": float((soc_arr < cfg.soc_min_safe).mean() * 100),
        "contention_pct": float(contended_steps / steps * 100) if steps else 0.0,
        "mean_info_age": float(np.mean(staleness_log)) if staleness_log else 0.0,
        "latency_mean": float(np.mean(completion_steps)) if completion_steps else float("nan"),
        "latency_median": float(np.median(completion_steps)) if completion_steps else float("nan"),
        "completion_steps": list(completion_steps),
        "blocked_moves": int(getattr(env, "n_blocked_moves", 0)),
    }


def summarize_runs(runs, level_value):
    """Collapse a list of per-seed metric dicts into mean/std statistics."""
    c = np.array([r["coverage_frac"] for r in runs]) * 100
    s = np.array([r["stranded_frac"] for r in runs]) * 100
    r = np.array([r["total_reward"] for r in runs])
    ex = np.array([r.get("expired_frac", 0.0) for r in runs]) * 100
    m = np.array([r["mean_final_soc"] for r in runs])
    st = np.array([r["steps_taken"] for r in runs])
    e = np.array([r["energy_wh_per_task"] for r in runs])
    d = np.array([r["dist_per_task_m"] for r in runs])
    mn = np.array([r["min_soc"] for r in runs])
    bf = np.array([r["below_floor_pct"] for r in runs])
    ct = np.array([r["contention_pct"] for r in runs])
    ia = np.array([r["mean_info_age"] for r in runs])
    lt = np.nan_to_num(np.array([r["latency_mean"] for r in runs], dtype=float), nan=float("nan"))
    lm = np.nan_to_num(np.array([r["latency_median"] for r in runs], dtype=float), nan=float("nan"))
    bm = np.array([r.get("blocked_moves", 0) for r in runs], dtype=float)
    def _ms(x):
        x = x[np.isfinite(x)]
        return (float(np.mean(x)), float(np.std(x))) if x.size else (0.0, 0.0)
    e_m, e_s = _ms(e); d_m, d_s = _ms(d); mn_m, mn_s = _ms(mn)
    bf_m, bf_s = _ms(bf); ct_m, ct_s = _ms(ct); ia_m, ia_s = _ms(ia)
    lt_m, lt_s = _ms(lt); lm_m, lm_s = _ms(lm)
    bm_m, bm_s = _ms(bm)
    return {
        "value": level_value,
        "coverage_mean": float(c.mean()), "coverage_std": float(c.std()),
        "stranded_mean": float(s.mean()), "stranded_std": float(s.std()),
        "reward_mean": float(r.mean()), "reward_std": float(r.std()),
        "expired_mean": float(ex.mean()), "expired_std": float(ex.std()),
        "soc_mean": float(m.mean()), "soc_std": float(m.std()),
        "steps_mean": float(st.mean()), "steps_std": float(st.std()),
        "energy_wh_per_task_mean": e_m, "energy_wh_per_task_std": e_s,
        "dist_per_task_m_mean": d_m, "dist_per_task_m_std": d_s,
        "min_soc_mean": mn_m, "min_soc_std": mn_s,
        "below_floor_pct_mean": bf_m, "below_floor_pct_std": bf_s,
        "contention_pct_mean": ct_m, "contention_pct_std": ct_s,
        "info_age_mean": ia_m, "info_age_std": ia_s,
        "latency_mean_mean": lt_m, "latency_mean_std": lt_s,
        "latency_median_mean": lm_m, "latency_median_std": lm_s,
        "blocked_moves_mean": bm_m, "blocked_moves_std": bm_s,
    }


def run_sweep(policy_fns, env_cfg, levels, level_field, n_seeds):
    """Sweep `level_field` (an EnvConfig3D attribute) across `levels`."""
    results = {}
    for name, fn in policy_fns.items():
        results[name] = []
        for level in levels:
            cfg = EnvConfig3D(**{**env_cfg.__dict__, level_field: level})
            runs = [run_episode(fn, cfg, s) for s in range(n_seeds)]
            results[name].append(summarize_runs(runs, level))
    return results


def run_fault_sweep(policy_fns, env_cfg, levels, n_seeds):
    """Robustness sweep: fail `frac` of the agents mid-episode."""
    results = {}
    for name, fn in policy_fns.items():
        results[name] = []
        for frac in levels:
            runs = [run_episode(fn, env_cfg, s, fail_frac=frac) for s in range(n_seeds)]
            results[name].append(summarize_runs(runs, frac))
    return results


def run_sweep_all_envs(policy_fns, env_cfgs, levels, level_field, n_seeds):
    """Run `run_sweep` independently in every evaluation environment."""
    out = {}
    for eid, cfg in env_cfgs.items():
        print(f"  sweep {level_field} @ {eid} ...", flush=True)
        out[eid] = run_sweep(policy_fns, cfg, levels, level_field, n_seeds)
    return out


def run_fault_sweep_all_envs(policy_fns, env_cfgs, levels, n_seeds):
    out = {}
    for eid, cfg in env_cfgs.items():
        print(f"  fault sweep @ {eid} ...", flush=True)
        out[eid] = run_fault_sweep(policy_fns, cfg, levels, n_seeds)
    return out


def run_base_all_envs(policy_fns, env_cfgs, n_seeds):
    """Base-config evaluation (8 UAVs, 8 tasks) in every environment."""
    results, runs_by_env = {}, {}
    for eid, cfg in env_cfgs.items():
        print(f"  base config @ {eid} ...", flush=True)
        runs_by_env[eid] = {
            name: [run_episode(fn, cfg, s) for s in range(n_seeds)]
            for name, fn in policy_fns.items()
        }
        results[eid] = {name: summarize_runs(rs, 0)
                        for name, rs in runs_by_env[eid].items()}
    return results, runs_by_env


## 3. MAPPO training (from scratch)

The actor/critic are built for a **maximum** task count (`N_TASKS_MAX`). During training
and evaluation, episodes with fewer tasks are zero-padded (extra task slots marked
inactive). This lets **one** trained policy act at 4 / 8 / 16 / 32 tasks, which is what
makes the task-density sweep meaningful for MAPPO.

Training uses the same recipe that made the 2D MAPPO tractable:

1. **Behavior-cloning warm-start** (`WARM_START`): the actor is cross-entropy cloned and the
   critic value-fitted onto Greedy-Nearest demonstration rollouts, replacing the cold random
   initialization. This is what stops the sparse 3D reward from collapsing the policy to
   "always hold".
2. **Training-only proximity shaping** (`SHAPING_BETA`): a dense bonus for being near active
   tasks is added to the PPO training signal (never to the reported metrics).
3. **Packet-loss curriculum**: the loss probability ramps from 0 up to `TARGET_PACKET_LOSS`.

A checkpoint is cached in `outputs/`, so re-running the notebook is fast.

In [ ]:
N_TASKS_MAX = max(32, ENV_CFG.n_tasks)
STRIDE = mt.TASK_FEAT_STRIDE


def build_padded_obs_mask(env):
    """Return (padded obs batch, padded action mask) for the current 3D state.
    Observation and action space are padded to N_TASKS_MAX; padded task slots
    are marked inactive (zero) and their actions masked out."""
    obs_dict = env._get_all_obs()
    obs_batch = np.stack([obs_dict[a] for a in env.agents])
    mask_batch = np.stack([env.action_mask(i) for i in range(env.n)])
    nt = env.n_tasks
    own_dim = mt.OWN_DIM
    ncap, ndim = env.cfg.neighbor_cap, mt.NEIGHBOR_DIM
    own = obs_batch[:, :own_dim]
    neigh = obs_batch[:, own_dim:own_dim + ncap * ndim]
    tasks = obs_batch[:, own_dim + ncap * ndim:]
    pad = (N_TASKS_MAX - nt) * STRIDE
    tasks_p = np.concatenate([tasks, np.zeros((obs_batch.shape[0], pad))], axis=1)
    obs_p = np.concatenate([own, neigh, tasks_p], axis=1)
    extra = N_TASKS_MAX - nt
    mask_p = np.concatenate(
        [mask_batch[:, :nt], np.zeros((obs_batch.shape[0], extra)),
         mask_batch[:, nt:]], axis=1)
    return obs_p, mask_p


def nearest_task_dist(env):
    """Per-agent 3D distance (m) to the nearest active task, clipped to SHAPING_DMAX."""
    tpos = env.task_pos[env.task_active]
    if len(tpos) == 0:
        return np.full(env.n, SHAPING_DMAX)
    d = np.linalg.norm(env.pos[:, None, :] - tpos[None, :, :], axis=-1)
    return np.min(d, axis=1).clip(None, SHAPING_DMAX)


def collect_rollout_vartask(env, actor, critic, n_episodes, rng, task_counts):
    """Collect a rollout while re-sampling the task count every episode, so the
    policy learns to act under varying task counts (variable-task curriculum).
    The stored reward adds a TRAINING-ONLY proximity shaping term; the env reward
    (used for all reported metrics) is left untouched."""
    out = {k: [] for k in ["own", "neigh_mask", "task", "actions",
                           "logprobs", "values", "rewards", "dones", "global"]}
    area = env.cfg.area_size
    alt = env.cfg.altitude_max
    for _ in range(n_episodes):
        env.cfg.n_tasks = int(rng.choice(task_counts))
        env.n_tasks = env.cfg.n_tasks
        env.action_space_size = env.cfg.n_tasks + 2
        env.RETURN_BASE = env.cfg.n_tasks
        env.HOLD = env.cfg.n_tasks + 1
        env.reset()
        done = False
        while not done:
            obs_p, mask_p = build_padded_obs_mask(env)
            own, neigh, valid_mask, tasks = mt.split_obs(
                obs_p, mt.OWN_DIM, env.cfg.neighbor_cap, mt.NEIGHBOR_DIM,
                area, alt)
            logits = actor.forward(own, neigh, valid_mask, tasks, mask_p)
            probs = softmax(logits)
            # stochastic sampling (not argmax) keeps exploration alive during
            # training; the evaluation policy below uses argmax (greedy)
            actions = mt.categorical_sample(probs, rng)
            logp = np.log(np.clip(probs[np.arange(env.n), actions], 1e-12, 1.0))
            gs = np.tile(mt.build_global_state(env), (env.n, 1))
            values = critic.forward(gs)
            action_dict = {a: int(actions[i]) for i, a in enumerate(env.agents)}
            _, reward_dict, term, trunc, _ = env.step(action_dict)
            rewards = np.array([reward_dict[a] for a in env.agents], dtype=np.float64)
            # training-only proximity shaping (not part of the reported reward)
            rewards = rewards + SHAPING_BETA * (1.0 - nearest_task_dist(env) / SHAPING_DMAX)
            done = all(term.values()) or all(trunc.values())
            out["own"].append(own); out["neigh_mask"].append((neigh, valid_mask))
            out["task"].append(tasks); out["actions"].append(actions)
            out["logprobs"].append(logp); out["values"].append(values)
            out["rewards"].append(rewards)
            out["dones"].append(np.full(env.n, float(done))); out["global"].append(gs)
    return out

In [ ]:
# ---- behavior-cloning warm-start helpers ---------------------------------
GAMMA_BC = 0.99


def collect_greedy_demos(env, n_episodes, rng, task_counts):
    """Roll out Greedy-Nearest in the 3D env and record every (padded obs,
    action) transition plus per-step returns-to-go, so the actor can be
    behavior-cloned and the critic value-fitted BEFORE any RL is run."""
    obs_all, act_all, mask_all, gs_all, ret_all = [], [], [], [], []
    for _ in range(n_episodes):
        env.cfg.n_tasks = int(rng.choice(task_counts))
        env.n_tasks = env.cfg.n_tasks
        env.action_space_size = env.cfg.n_tasks + 2
        env.RETURN_BASE = env.cfg.n_tasks
        env.HOLD = env.cfg.n_tasks + 1
        env.reset()
        done = False
        steps = []
        while not done:
            obs_p, mask_p = build_padded_obs_mask(env)
            gs = np.tile(mt.build_global_state(env), (env.n, 1))
            act_dict = greedy_nearest_policy(env)
            acts = np.array([act_dict[a] for a in env.agents], dtype=int)
            _, reward_dict, term, trunc, _ = env.step(act_dict)
            rewards = np.array([reward_dict[a] for a in env.agents], dtype=np.float64)
            steps.append((obs_p, acts, mask_p, gs, rewards))
            done = all(term.values()) or all(trunc.values())
        G = np.zeros(env.n)
        for obs_p, acts, mask_p, gs, rewards in reversed(steps):
            G = rewards + GAMMA_BC * G
            obs_all.append(obs_p); act_all.append(acts); mask_all.append(mask_p)
            gs_all.append(gs); ret_all.append(np.tile(G, (env.n, 1)))
    return (np.concatenate(obs_all), np.concatenate(act_all), np.concatenate(mask_all),
            np.concatenate(gs_all), np.concatenate(ret_all))


def bc_pretrain(actor, critic, actor_opt, critic_opt, data, iters, rng, batch=128):
    """Behavior-clone the actor onto the demonstrated actions (cross-entropy) and
    value-fit the critic onto the demonstrated returns-to-go. Both networks are
    warm-started this way, so the subsequent PPO fine-tune starts from a
    task-competent policy instead of a cold random one."""
    obs_all, act_all, mask_all, gs_all, ret_all = data
    n = obs_all.shape[0]
    losses = []
    for it in range(iters):
        idx = rng.integers(0, n, size=batch)
        own, neigh, valid_mask, tasks = mt.split_obs(
            obs_all[idx], mt.OWN_DIM, ENV_CFG.neighbor_cap, mt.NEIGHBOR_DIM,
            ENV_CFG.area_size, ENV_CFG.altitude_max)
        logits = actor.forward(own, neigh, valid_mask, tasks, mask_all[idx])
        probs = softmax(logits)
        logp = np.log(np.clip(probs[np.arange(batch), act_all[idx]], 1e-12, 1.0))
        losses.append(-float(np.mean(logp)))
        onehot = np.zeros_like(logits)
        onehot[np.arange(batch), act_all[idx]] = 1.0
        actor_grads = actor.backward((probs - onehot) / batch)
        actor_param_grads = []
        for _name, _layer in actor.all_layers().items():
            _g = actor_grads[_name]
            actor_param_grads.append(_g["W"]); actor_param_grads.append(_g["b"])
        actor_opt.step(actor_param_grads)
        cidx = rng.integers(0, n, size=batch)
        gs = gs_all[cidx]
        tgt = ret_all[cidx][:, 0]
        dvalue = 2.0 * (critic.forward(gs) - tgt) / batch
        critic_grads = critic.backward(dvalue)
        critic_param_grads = []
        for _name, _layer in critic.all_layers().items():
            _g = critic_grads[1][_name]
            critic_param_grads.append(_g["W"]); critic_param_grads.append(_g["b"])
        critic_opt.step(critic_param_grads)
        if (it + 1) % 1000 == 0:
            print(f"  BC iter {it+1} loss={losses[-1]:.3f}", flush=True)
    return losses

In [ ]:
def clear_caches(o, _seen=None):
    # Null out all forward-pass caches so pickled checkpoints stay tiny.
    # Recurses into containers (dict/list/tuple) as well as objects, because
    # the MARL MLP keeps its Linear layers in nested lists.
    _seen = _seen if _seen is not None else set()
    if o is None or isinstance(o, (str, bytes, int, float, bool)) or id(o) in _seen:
        return
    _seen.add(id(o))
    if hasattr(o, "__dict__"):
        for _name, _val in list(vars(o).items()):
            if _name == "_cache":
                vars(o)["_cache"] = None
            else:
                clear_caches(_val, _seen)
    elif isinstance(o, dict):
        for _val in o.values():
            clear_caches(_val, _seen)
    elif isinstance(o, (list, tuple)):
        for _val in o:
            clear_caches(_val, _seen)


CKPT = os.path.join(OUT_DIR, f"mappo_{CFG_NAME}_{SCENARIO}_{N_TRAIN_EPISODES}ep.pkl")
train_env = UAVSwarmEnv3D(EnvConfig3D(**{**ENV_CFG.__dict__, "seed": SEED}))

if os.path.exists(CKPT):
    with open(CKPT, "rb") as f:
        d = pickle.load(f)
    actor, critic, log = d["actor"], d["critic"], d["log"]
    trained_episodes = d["n_episodes"]
    print(f"Loaded existing checkpoint trained for {trained_episodes} episodes "
          f"-> {CKPT}")
else:
    rng = np.random.default_rng(SEED)
    actor = Actor(mt.OWN_DIM, mt.NEIGHBOR_DIM, ENV_CFG.neighbor_cap,
                  N_TASKS_MAX * STRIDE, N_TASKS_MAX + 2, hidden_dim=ACTOR_HIDDEN, rng=rng)
    critic = Critic(5 * ENV_CFG.n_agents, hidden_dim=64, rng=rng)
    actor_opt, critic_opt = mt.make_actor_critic_optims(actor, critic, lr=1e-3)

    log = []
    t0 = time.time()
    if WARM_START:
        # 1) behavior-clone the actor (and value-fit the critic) onto Greedy-Nearest
        #    rollouts -- this replaces the cold random initialization and is what
        #    lets the from-scratch NumPy PPO reach the engineered baselines here.
        print(f"Collecting {BC_EPISODES} Greedy-Nearest demo episodes...", flush=True)
        demo = collect_greedy_demos(train_env, BC_EPISODES, rng, TRAIN_TASK_COUNTS)
        print(f"  -> {demo[0].shape[0]} transitions; BC pre-train {BC_ITERS} iters...",
              flush=True)
        bc_losses = bc_pretrain(actor, critic, actor_opt, critic_opt, demo,
                                BC_ITERS, rng)
        log.append({"stage": "bc", "episodes": BC_EPISODES, "iters": BC_ITERS,
                    "final_bc_loss": float(bc_losses[-1])})
        actor_opt, critic_opt = mt.make_actor_critic_optims(actor, critic, lr=3e-4)

    episode = 0
    while episode < N_TRAIN_EPISODES:
        train_env.cfg.packet_loss_prob = TARGET_PACKET_LOSS * min(
            1.0, episode / max(CURRICULUM_EPISODES, 1))
        buf = collect_rollout_vartask(train_env, actor, critic,
                                      PPO_EPISODES_PER_UPDATE, rng, TRAIN_TASK_COUNTS)
        stats = mt.ppo_update(actor, critic, actor_opt, critic_opt, buf, ent_coef=0.02)
        episode += PPO_EPISODES_PER_UPDATE
        log.append({"episode": episode,
                    "packet_loss": float(train_env.cfg.packet_loss_prob),
                    "mean_reward": stats["mean_reward"],
                    "entropy": stats["entropy"],
                    "value_loss": stats["value_loss"]})
        print(f"[ep {episode:5d}] p_loss={train_env.cfg.packet_loss_prob:.2f} "
              f"mean_reward={stats['mean_reward']:7.3f} entropy={stats['entropy']:.3f} "
              f"value_loss={stats['value_loss']:.4f}  ({time.time()-t0:.0f}s)",
              flush=True)
    trained_episodes = episode
    clear_caches(actor)
    clear_caches(critic)
    with open(CKPT, "wb") as f:
        pickle.dump({"actor": actor, "critic": critic, "log": log,
                     "cfg": ENV_CFG, "n_episodes": episode}, f)
    print(f"\nTraining done in {time.time()-t0:.0f}s for {episode} episodes; "
          f"checkpoint -> {CKPT}")

## 3.5 Off-policy MARL baselines: MADDPG and QMIX

Both baselines are trained **from the same offline replay buffer** of greedy-style
demonstration episodes (with light eps-greedy exploration so RETURN / HOLD are actually
seen in the data), behaviour-cloned first and then fine-tuned with replay RL updates --
the same BC warm-start recipe that makes the MAPPO baseline tractable in NumPy. Training
is centralised (MADDPG's centralized critic, QMIX's global-state mixing network), but both
policies **execute fully decentralized** (per-agent actor argmax / per-agent utility
argmax), so they are fair CTDE baselines. Checkpoints are cached in `outputs/` so
re-running the notebook does not retrain them.

In [ ]:
def collect_mar_demos(n_episodes, task_counts, rng, eps=0.0):
    """Roll out Greedy-Nearest (with optional eps-greedy exploration) and record
    every (padded obs, action, reward, next obs, masks, done, state, next_state)
    transition -- the shared offline replay buffer for MADDPG and QMIX."""
    buf = ReplayBuffer(120000)
    _env = UAVSwarmEnv3D(EnvConfig3D(**{**ENV_CFG.__dict__, "seed": SEED}))
    for _ in range(n_episodes):
        _env.cfg.n_tasks = int(rng.choice(task_counts))
        _env.n_tasks = _env.cfg.n_tasks
        _env.action_space_size = _env.cfg.n_tasks + 2
        _env.RETURN_BASE = _env.cfg.n_tasks
        _env.HOLD = _env.cfg.n_tasks + 1
        _env.reset()
        done = False
        while not done:
            obs_p, mask_p = build_padded_obs_mask(_env)
            obs_p = normalize_padded_obs(obs_p, ENV_CFG.area_size,
                                         ENV_CFG.altitude_max,
                                         ENV_CFG.neighbor_cap, N_TASKS_MAX)
            state = build_state(_env)
            g = np.array([greedy_nearest_policy(_env)[a] for a in _env.agents],
                         dtype=int)
            act = g.copy()
            if eps > 0:
                for i in range(_env.n):
                    if rng.random() < eps:
                        valid = np.where(mask_p[i] > 0.5)[0]
                        act[i] = int(rng.choice(valid))
            _, reward_dict, term, trunc, _ = _env.step(
                {a: int(act[i]) for i, a in enumerate(_env.agents)})
            next_obs_p, next_mask_p = build_padded_obs_mask(_env)
            next_obs_p = normalize_padded_obs(next_obs_p, ENV_CFG.area_size,
                                              ENV_CFG.altitude_max,
                                              ENV_CFG.neighbor_cap, N_TASKS_MAX)
            next_state = build_state(_env)
            rew = np.array([reward_dict[a] for a in _env.agents], dtype=np.float64)
            done = all(term.values()) or all(trunc.values())
            buf.add(obs_p, act, rew, next_obs_p, mask_p, next_mask_p, done,
                    state, next_state)
    return buf


def mar_demo_stack(buf):
    """Flattened (obs, act, mask) arrays over the whole buffer for BC."""
    obs, act, mask = [], [], []
    for b in buf.buf:
        obs.append(b[0]); act.append(b[1]); mask.append(b[4])
    return np.concatenate(obs), np.concatenate(act), np.concatenate(mask)


OBS_DIM = 4 + ENV_CFG.neighbor_cap * 6 + N_TASKS_MAX * STRIDE
N_ACTIONS = N_TASKS_MAX + 2
STATE_DIM = ENV_CFG.n_agents * 5


def _load_meta_int(name, key):
    # Read an integer field from a small JSON sidecar in OUT_DIR (None if absent).
    p = os.path.join(OUT_DIR, name)
    if not os.path.exists(p):
        return None
    try:
        with open(p) as f:
            return json.load(f).get(key)
    except Exception:
        return None



def maddpg_policy(env):
    """Decentralized MADDPG eval policy: per-agent actor argmax."""
    o, m = build_padded_obs_mask(env)
    acts, _ = maddpg.act(normalize_padded_obs(o, ENV_CFG.area_size,
                                              ENV_CFG.altitude_max,
                                              ENV_CFG.neighbor_cap, N_TASKS_MAX),
                         m, rng=None, greedy=True)
    return {env.agents[i]: int(acts[i]) for i in range(env.n)}


def qmix_policy(env):
    """Decentralized QMIX eval policy: per-agent utility argmax."""
    o, m = build_padded_obs_mask(env)
    acts = qmix.act(normalize_padded_obs(o, ENV_CFG.area_size,
                                         ENV_CFG.altitude_max,
                                         ENV_CFG.neighbor_cap, N_TASKS_MAX),
                    m, greedy=True)
    return {env.agents[i]: int(acts[i]) for i in range(env.n)}


CKPT_MADDPG = os.path.join(OUT_DIR, f"maddpg_{CFG_NAME}.pkl")
CKPT_QMIX = os.path.join(OUT_DIR, f"qmix_{CFG_NAME}.pkl")

rng_mar = np.random.default_rng(SEED + 1)
mar_buf = None
if os.path.exists(CKPT_MADDPG) and os.path.exists(CKPT_QMIX):
    with open(CKPT_MADDPG, "rb") as f:
        maddpg = pickle.load(f)
    with open(CKPT_QMIX, "rb") as f:
        qmix = pickle.load(f)
    print(f"Loaded existing MADDPG/QMIX checkpoints -> {CKPT_MADDPG}, {CKPT_QMIX}")
else:
    t0 = time.time()
    print(f"Collecting {MARL_DEMO_EPISODES} demo episodes "
          f"(eps={MARL_EPS_DEMO})...", flush=True)
    mar_buf = collect_mar_demos(MARL_DEMO_EPISODES, MARL_TASK_COUNTS,
                                rng_mar, eps=MARL_EPS_DEMO)
    with open(os.path.join(OUT_DIR, "mar_demo_meta.json"), "w") as f:
        json.dump({"n_transitions": len(mar_buf.buf)}, f)
    print(f"  -> {len(mar_buf)} transitions in {time.time()-t0:.0f}s")

    # ---- MADDPG: BC warm-start the shared actor, then RL fine-tune --------
    t0 = time.time()
    maddpg = MADDPG(OBS_DIM, N_ACTIONS, ENV_CFG.n_agents, hidden=32,
                    lr=MADDPG_LR, seed=SEED)
    obs_all, act_all, mask_all = mar_demo_stack(mar_buf)
    losses = bc_warmstart_actor(maddpg, obs_all, act_all, mask_all,
                                MARL_BC_ITERS, np.random.default_rng(SEED + 2),
                                batch=128)
    print(f"MADDPG BC warm-start done ({len(losses)} iters, final loss "
          f"{losses[-1]:.4f}) in {time.time()-t0:.0f}s", flush=True)
    t0 = time.time()
    for it in range(MARL_RL_ITERS):
        st = update_from_buffer(maddpg, mar_buf, 128,
                                np.random.default_rng(100 + it))
        if it % 500 == 0:
            print(f"  MADDPG it {it}: cL={st['critic_loss']:.3f} "
                  f"pL={st['policy_loss']:.3f} ent={st['entropy']:.3f} "
                  f"q={st['mean_q']:.2f}", flush=True)
    clear_caches(maddpg)
    with open(CKPT_MADDPG, "wb") as f:
        pickle.dump(maddpg, f)
    print(f"MADDPG done in {time.time()-t0:.0f}s -> {CKPT_MADDPG}")

    # ---- QMIX: reward-normalized copy of the same buffer + BC-regularized TD
    maxabs = max(abs(b[2]).max() for b in mar_buf.buf)
    qbuf = ReplayBuffer(len(mar_buf.buf))
    for b in mar_buf.buf:
        o, a, r, no, m, nm, d, s, ns = b
        qbuf.buf.append((o, a, r / maxabs, no, m, nm, d, s, ns))
    t0 = time.time()
    qmix = QMIX(OBS_DIM, N_ACTIONS, ENV_CFG.n_agents, STATE_DIM, hidden=32,
                mix_hidden=32, lr=QMIX_LR, gamma=0.99, seed=SEED,
                target_update_every=QMIX_TARGET_EVERY)
    obs_all, act_all, mask_all = mar_demo_stack(qbuf)
    losses = qmix_bc_warmstart(qmix, obs_all, act_all, mask_all,
                               MARL_BC_ITERS, np.random.default_rng(SEED + 3),
                               batch=128)
    print(f"QMIX BC warm-start done ({len(losses)} iters, final loss "
          f"{losses[-1]:.4f}) in {time.time()-t0:.0f}s", flush=True)
    t0 = time.time()
    for it in range(MARL_RL_ITERS):
        sample = qbuf.sample(128, np.random.default_rng(200 + it))
        st = qmix_update(qmix, sample, 128, np.random.default_rng(300 + it),
                         q_lim=QMIX_Q_LIM, bc_weight=QMIX_BC_WEIGHT)
        if it % 500 == 0:
            print(f"  QMIX it {it}: td={st['td_loss']:.1e} "
                  f"qtot={st['mean_q_tot']:.2f} bc={st['bc_loss']:.3f}",
                  flush=True)
    clear_caches(qmix)
    with open(CKPT_QMIX, "wb") as f:
        pickle.dump(qmix, f)
    print(f"QMIX done in {time.time()-t0:.0f}s -> {CKPT_QMIX}")

maddpg_log = {"config": CFG_NAME, "scenario": SCENARIO,
              "demo_episodes": MARL_DEMO_EPISODES,
              "eps_demo": MARL_EPS_DEMO, "bc_iters": MARL_BC_ITERS,
              "rl_iters": MARL_RL_ITERS,
              "n_transitions": (len(mar_buf.buf) if mar_buf else
                                _load_meta_int("mar_demo_meta.json",
                                               "n_transitions"))}
print("MADDPG/QMIX ready.")

In [ ]:
def mappo_policy(env):
    """Deterministic MAPPO action dict for the current 3D env state. Works for
    any task count <= N_TASKS_MAX via zero-padding."""
    obs_p, mask_p = build_padded_obs_mask(env)
    own, neigh, valid_mask, tasks = mt.split_obs(
        obs_p, mt.OWN_DIM, env.cfg.neighbor_cap, mt.NEIGHBOR_DIM,
        env.cfg.area_size, env.cfg.altitude_max)
    probs = softmax(actor.forward(own, neigh, valid_mask, tasks, mask_p))
    acts = np.argmax(probs, axis=1)
    return {a: int(acts[i]) for i, a in enumerate(env.agents)}


# --- HRL-H (proposed model) + its two ablations --------------------------
def mappo_prefer_fn(env):
    """High-level allocation scores from the MAPPO actor: task-slot marginal
    probabilities (columns 0..N_TASKS_MAX-1)."""
    obs_p, mask_p = build_padded_obs_mask(env)
    own, neigh, valid_mask, tasks = mt.split_obs(
        obs_p, mt.OWN_DIM, env.cfg.neighbor_cap, mt.NEIGHBOR_DIM,
        env.cfg.area_size, env.cfg.altitude_max)
    probs = softmax(actor.forward(own, neigh, valid_mask, tasks, mask_p))
    return probs[:, :N_TASKS_MAX]


def qmix_prefer_fn(env):
    """High-level allocation scores from the QMIX q-network: per-agent task
    utilities with padded (inactive) slots set to -inf."""
    obs_p, mask_p = build_padded_obs_mask(env)
    q = qmix.qnet.forward(normalize_padded_obs(
        obs_p, ENV_CFG.area_size, ENV_CFG.altitude_max,
        ENV_CFG.neighbor_cap, N_TASKS_MAX))
    q = q + np.where(mask_p > 0.5, 0.0, -1e9)
    scores = np.full((env.n, N_TASKS_MAX), -np.inf)
    scores[:, :env.n_tasks] = q[:, :env.n_tasks]
    return scores


hrlh = HRLH(mappo_prefer_fn, with_navigation=True)          # proposed model
hrlh_nav = HRLH(mappo_prefer_fn, with_navigation=False,
                raw_policy_fn=mappo_policy)                  # ablation 1
hrlh_q = HRLH(qmix_prefer_fn, with_navigation=True)          # ablation 2

ABLATIONS = {"hrlh_nav": hrlh_nav, "hrlh_qmix": hrlh_q}

POLICIES = {
    "random": random_policy,
    "greedy": greedy_nearest_policy,
    "cbba": cbba_policy,
    "mappo": mappo_policy,
    "maddpg": maddpg_policy,
    "qmix": qmix_policy,
    "dmpc": dmpc_policy,
    "hrlh": hrlh,
}

# --- sanity: one episode per policy at the base config --------------------
for name, fn in POLICIES.items():
    r = run_episode(fn, ENV_CFG, seed=SEED)
    print(f"{name:<10} reward={r['total_reward']:8.1f}  "
          f"coverage={r['coverage_frac'] * 100:5.1f}%  stranded={r['n_stranded']}  "
          f"steps={r['steps_taken']}")

In [ ]:
def save_fig(fig, name, extra_dir=None):
    """Save a figure as a 600 dpi PNG in figures/ (and optionally a subfolder)."""
    path = os.path.join(FIG_DIR, f"{CFG_NAME}_{name}.png")
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    if extra_dir is not None:
        os.makedirs(extra_dir, exist_ok=True)
        fig.savefig(os.path.join(extra_dir, f"{CFG_NAME}_{name}.png"),
                    dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print("saved", path)


def _style_ax(ax, xlabel, ylabel, title=None):
    ax.set_xlabel(xlabel, fontsize=FONT_LABEL)
    ax.set_ylabel(ylabel, fontsize=FONT_LABEL)
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)


def plot_training_curve(log, fname):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
    ppo = [l for l in log if "episode" in l]
    for ax, key, label in zip(axes, ["mean_reward", "entropy", "value_loss"],
                              ["Mean reward (per agent-step)", "Policy entropy",
                               "Value loss"]):
        ax.plot([l["episode"] for l in ppo], [l[key] for l in ppo],
                color=POLICY_COLORS["mappo"], linewidth=2.4, label="MAPPO")
        _style_ax(ax, "Training episode", label, label)
        ax.legend(loc="best", fontsize=FONT_LEGEND, frameon=True)
    fig.suptitle(f"MAPPO training curves -- 3D {CFG_NAME} config",
                 fontsize=FONT_SUPTITLE)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    save_fig(fig, fname)


### MAPPO training curves + training log

In [ ]:
plot_training_curve(log, "mappo_training_curve")
with open(os.path.join(OUT_DIR, "mappo_training.json"), "w") as f:
    json.dump({"config": CFG_NAME, "scenario": SCENARIO,
               "n_episodes": trained_episodes,
               "task_counts": TRAIN_TASK_COUNTS, "log": log}, f, indent=1)
print("saved outputs/mappo_training.json")

## 4. Environment snapshots (2x2: open, layered, open+NFZ, layered+NFZ)

A shared seed is used so agent/task XY layout is comparable; the dedicated obstacle
RNG stream keeps the layout identical when no-fly zones are toggled. Each snapshot is
saved as JSON under `outputs/env_snapshots/` so the 2x2 figure can be re-plotted later
without re-running the environment.


In [ ]:
SNAP_DIR = os.path.join(OUT_DIR, "env_snapshots")
os.makedirs(SNAP_DIR, exist_ok=True)

env_snapshots = []
for eid, label, scenario, obs_on, n_obs in ENV_SPECS:
    cfg = EnvConfig3D(**{**ENV_CFGS[eid].__dict__, "seed": 3})
    env0 = UAVSwarmEnv3D(cfg)
    env0.reset()
    snap = collect_env_snapshot(env0, eid, label)
    env_snapshots.append(snap)
    save_env_snapshot(snap, os.path.join(SNAP_DIR, f"{eid}.json"))
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")
    plot_env_3d(env0, f"{label} ({CFG_NAME})", ax=ax)
    fig.tight_layout()
    save_fig(fig, f"env_snapshot_{eid}")
    print(f"snapshot {eid}: agents={env0.n} tasks={env0.n_tasks} "
          f"obstacles={len(env0.obstacles)}")

write_json({"config": CFG_NAME, "seed": 3, "environments": ENV_ORDER,
            "snapshots": {s["env_id"]: s for s in env_snapshots}},
           os.path.join(SNAP_DIR, "all_environments.json"))

fig = plot_environments_2x2(
    env_snapshots,
    title=f"3D evaluation environments -- 2x2 ({CFG_NAME} config)",
    figsize=(18, 16),
)
save_fig(fig, "env_snapshot_2x2")
print("saved 2x2 environment plot + per-environment snapshot JSON")

# Rebuild the 2x2 later from JSON only (no env reset required):
# fig = replot_environments_2x2_from_dir(SNAP_DIR)
# save_fig(fig, "env_snapshot_2x2")


## 5. Evaluation: packet-loss sweep (all 4 environments)

Levels: **0%, 20%, 40%, 60%**. Results are written per environment as JSON + CSV
(no sweep plots -- use the comparison tables).


In [ ]:
PL_LEVELS = [0.0, 0.2, 0.4, 0.6]
print("packet-loss sweep across", ENV_ORDER)
pl_results_by_env = run_sweep_all_envs(
    POLICIES, ENV_CFGS, PL_LEVELS, "packet_loss_prob", N_SEEDS)
dump_sweep_by_env(pl_results_by_env, OUT_DIR, "eval_packet_loss",
                  {"config": CFG_NAME, "n_seeds": N_SEEDS, "levels": PL_LEVELS,
                   "sweep": "packet_loss"})
# keep a layered alias for backward compatibility
pl_results = pl_results_by_env["layered"]
print("saved outputs/eval_packet_loss.json + .csv (per environment)")


## 6. Evaluation: communication-range sweep (all 4 environments)

Levels: **6, 12, 25, 50 m**. Saved per environment as JSON + CSV (tables, no graphs).


In [ ]:
CR_LEVELS = [6.0, 12.0, 25.0, 50.0]
print("comm-range sweep across", ENV_ORDER)
cr_results_by_env = run_sweep_all_envs(
    POLICIES, ENV_CFGS, CR_LEVELS, "comm_range", N_SEEDS)
dump_sweep_by_env(cr_results_by_env, OUT_DIR, "eval_comm_range",
                  {"config": CFG_NAME, "n_seeds": N_SEEDS, "levels": CR_LEVELS,
                   "sweep": "comm_range"})
cr_results = cr_results_by_env["layered"]
print("saved outputs/eval_comm_range.json + .csv (per environment)")


## 7. Evaluation: task-density sweep (all 4 environments)

Levels: **4, 8, 16, 32 tasks**. Saved per environment as JSON + CSV (tables, no graphs).


In [ ]:
TD_LEVELS = [4, 8, 16, 32]
print("task-density sweep across", ENV_ORDER)
td_results_by_env = run_sweep_all_envs(
    POLICIES, ENV_CFGS, TD_LEVELS, "n_tasks", N_SEEDS)
dump_sweep_by_env(td_results_by_env, OUT_DIR, "eval_task_density",
                  {"config": CFG_NAME, "n_seeds": N_SEEDS, "levels": TD_LEVELS,
                   "sweep": "task_density"})
td_results = td_results_by_env["layered"]
print("saved outputs/eval_task_density.json + .csv (per environment)")


## 7.5 Evaluation: UAV-dropout / fault-tolerance sweep (all 4 environments)

Levels: **0%, 25%, 50%**. Saved per environment as JSON + CSV (tables, no graphs).


In [ ]:
FT_LEVELS = [0.0, 0.25, 0.5]
print("UAV-dropout sweep across", ENV_ORDER)
ft_results_by_env = run_fault_sweep_all_envs(
    POLICIES, ENV_CFGS, FT_LEVELS, N_SEEDS)
dump_sweep_by_env(ft_results_by_env, OUT_DIR, "eval_fault_tolerance",
                  {"config": CFG_NAME, "n_seeds": N_SEEDS, "levels": FT_LEVELS,
                   "sweep": "fault_tolerance"})
ft_results = ft_results_by_env["layered"]
print("saved outputs/eval_fault_tolerance.json + .csv (per environment)")


## 7.6 Evaluation: UAV-count / scalability sweep (all 4 environments)

Levels: **4, 8, 16 UAVs**. Saved per environment as JSON + CSV (tables, no graphs).


In [ ]:
SC_LEVELS = [4, 8, 16]
print("UAV-count sweep across", ENV_ORDER)
sc_results_by_env = {}
for eid, env_cfg in ENV_CFGS.items():
    print(f"  scalability @ {eid} ...", flush=True)
    sc_results_by_env[eid] = {}
    for name, fn in POLICIES.items():
        sc_results_by_env[eid][name] = []
        for n in SC_LEVELS:
            cfg = EnvConfig3D(**{**env_cfg.__dict__, "n_agents": n, "n_tasks": n})
            runs = [run_episode(fn, cfg, s) for s in range(N_SEEDS)]
            sc_results_by_env[eid][name].append(summarize_runs(runs, n))
dump_sweep_by_env(sc_results_by_env, OUT_DIR, "eval_scalability",
                  {"config": CFG_NAME, "n_seeds": N_SEEDS, "levels": SC_LEVELS,
                   "sweep": "scalability"})
sc_results = sc_results_by_env["layered"]
print("saved outputs/eval_scalability.json + .csv (per environment)")


## 8. Evaluation: 4-environment comparison (open / layered / NFZ)

Every approach runs at the **base configuration** (8 UAVs, 8 tasks) in all four
environments. Grouped bars show coverage, stranded UAVs, and total reward. Results
are also the source of the comparison tables saved at the end of the notebook.


In [ ]:
def plot_scenario_comparison(results_by_env, env_ids, fname):
    """One row per environment, 3 metric columns -- 4x3 grid with large labels."""
    n_env = len(env_ids)
    fig, axes = plt.subplots(n_env, 3, figsize=(24, 5.4 * n_env))
    metrics = [("coverage_mean", "Coverage (%)", "coverage_std"),
               ("stranded_mean", "Stranded UAVs (%)", "stranded_std"),
               ("reward_mean", "Total reward", "reward_std")]
    x = np.arange(len(POLICY_ORDER)); width = 0.62
    for r, eid in enumerate(env_ids):
        for c, (mk, label, sk) in enumerate(metrics):
            ax = axes[r, c]
            vals = [results_by_env[eid][name][mk] for name in POLICY_ORDER]
            errs = [results_by_env[eid][name][sk] for name in POLICY_ORDER]
            bars = ax.bar(x, vals, width, yerr=errs, capsize=5,
                          color=[POLICY_COLORS[n] for n in POLICY_ORDER],
                          edgecolor="k", linewidth=0.5)
            for b, v in zip(bars, vals):
                ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}",
                        ha="center", va="bottom", fontsize=FONT_ANNOT)
            ax.set_xticks(x)
            ax.set_xticklabels([POLICY_LABELS[n] for n in POLICY_ORDER],
                               fontsize=FONT_TICK - 1, rotation=22, ha="right")
            ax.set_ylabel(label, fontsize=FONT_LABEL)
            ax.set_title(f"{ENV_LABELS.get(eid, eid)} -- {label}",
                         fontsize=FONT_TITLE)
            ax.tick_params(labelsize=FONT_TICK)
    fig.suptitle(f"3D environment comparison -- all 8 approaches ({CFG_NAME})",
                 fontsize=FONT_SUPTITLE)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    save_fig(fig, fname)


print("base-config evaluation across", ENV_ORDER)
scenario_results, scenario_runs = run_base_all_envs(POLICIES, ENV_CFGS, N_SEEDS)
plot_scenario_comparison(scenario_results, ENV_ORDER, "scenario_comparison")
dump_base_by_env(scenario_results, OUT_DIR, "eval_scenarios",
                 {"config": CFG_NAME, "n_seeds": N_SEEDS,
                  "environments": ENV_ORDER, "env_labels": ENV_LABELS})
print("saved outputs/eval_scenarios.json + .csv (per environment)")


## 8.5 Trajectories (all eight approaches, 3D): four environments

Trajectories are recorded in **all four environments** and each is rendered as a
**4x2 grid**, one 3D panel per approach. UAV paths are colored by **final SOC**
(red = nearly stranded, green = healthy). Per-policy metrics are exported as
**JSON and CSV** under `outputs/trajectories_<env_id>.json/.csv`, plus a combined
`trajectories.csv` with an `environment` column so every model can be compared
across open / layered / NFZ later without re-running.


In [ ]:
def record_episode(policy_fn, env_cfg, seed):
    cfg = EnvConfig3D(**{**env_cfg.__dict__, "seed": seed})
    env = UAVSwarmEnv3D(cfg)
    env.reset()
    traj = [env.pos.copy()]
    soc_hist = [env.soc.copy()]
    for t in range(cfg.max_steps):
        actions = policy_fn(env)
        _, _, term, trunc, _ = env.step(actions)
        traj.append(env.pos.copy())
        soc_hist.append(env.soc.copy())
        if all(term.values()) or all(trunc.values()):
            break
    return env, np.stack(traj), np.stack(soc_hist)


def plot_trajectories_3d(records, fname, title_tag):
    """4 rows x 2 columns grid (one panel per approach) of 3D trajectories."""
    fig = plt.figure(figsize=(26, 40))
    for idx, name in enumerate(POLICY_ORDER, start=1):
        ax = fig.add_subplot(4, 2, idx, projection="3d")
        env, traj, soc_hist = records[name]
        ax.scatter(*env.base_pos, marker="s", s=140, c="black",
                   label="Base / depot", depthshade=False)
        active = env.task_active
        if active.any():
            ax.scatter(env.task_pos[active, 0], env.task_pos[active, 1],
                       env.task_pos[active, 2], marker="*", s=130, c="orange",
                       edgecolors="k", label="Active task", depthshade=False)
        if (~active).any():
            ax.scatter(env.task_pos[~active, 0], env.task_pos[~active, 1],
                       env.task_pos[~active, 2], marker="x", s=50, c="lightgray",
                       label="Completed task", depthshade=False)
        colors = plt.cm.RdYlGn(np.clip(soc_hist[-1], 0, 1))
        for i in range(env.n):
            ax.plot(traj[:, i, 0], traj[:, i, 1], traj[:, i, 2], "-",
                    color=colors[i], alpha=0.85, linewidth=1.4)
            ax.scatter(*traj[0, i], marker="o", s=35, c="gray", depthshade=False)
            marker_end = "X" if env.stranded[i] else "^"
            ax.scatter(*traj[-1, i], marker=marker_end, s=100, c=[colors[i]],
                       edgecolors="k", depthshade=False)
        obs = getattr(env, "obstacles", None)
        if obs is not None and len(obs):
            from mpl_toolkits.mplot3d.art3d import Poly3DCollection
            for o in obs:
                x0, y0, z0, x1, y1, z1 = o
                corners = [
                    [(x0, y0, z0), (x1, y0, z0), (x1, y1, z0), (x0, y1, z0)],
                    [(x0, y0, z1), (x1, y0, z1), (x1, y1, z1), (x0, y1, z1)],
                ]
                for x in (x0, x1):
                    corners.append([(x, y0, z0), (x, y1, z0), (x, y1, z1), (x, y0, z1)])
                for y in (y0, y1):
                    corners.append([(x0, y, z0), (x1, y, z0), (x1, y, z1), (x0, y, z1)])
                ax.add_collection3d(Poly3DCollection(
                    corners, alpha=0.15, facecolor="#b30000", edgecolor="#7a0000",
                    linewidths=0.5, label="No-fly zone" if o is obs[0] else None))
        L = env.cfg.area_size
        xx, yy = np.meshgrid([0, L], [0, L])
        ax.plot_surface(xx, yy, np.zeros_like(xx), alpha=0.05, color="gray")
        ax.set_xlabel("x (m)", fontsize=FONT_LABEL)
        ax.set_ylabel("y (m)", fontsize=FONT_LABEL)
        ax.set_zlabel("altitude z (m)", fontsize=FONT_LABEL)
        ax.tick_params(labelsize=FONT_TICK)
        ax.set_zlim(0, env.cfg.altitude_max * 1.1)
        ax.set_title(POLICY_LABELS[name], fontsize=FONT_TITLE)
        ax.legend(loc="upper left", fontsize=FONT_LEGEND)
    fig.suptitle(f"3D trajectories {title_tag} -- all 8 approaches ({CFG_NAME})\n"
                 "trail color = final SOC (red = nearly stranded, green = healthy)",
                 fontsize=FONT_SUPTITLE)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    save_fig(fig, fname)


def summarize_traj(env, traj, environment):
    return {
        "environment": environment,
        "scenario": env.cfg.scenario,
        "obstacles_enabled": bool(env.cfg.obstacles_enabled),
        "n_obstacles": int(env.cfg.n_obstacles),
        "tasks_completed": int((~env.task_active).sum()),
        "tasks_total": int(env.n_tasks),
        "coverage_pct": float((~env.task_active).sum()) / env.n_tasks * 100.0,
        "n_stranded": int(env.stranded.sum()),
        "mean_final_soc": float(env.soc.mean()),
        "steps_taken": int(traj.shape[0] - 1),
        "blocked_moves": int(getattr(env, "n_blocked_moves", 0)),
        "trajectory": traj.tolist(),
        "final_soc": env.soc.tolist(),
    }


def record_and_plot(policy_fns, env_cfg, tag, environment):
    records, rows = {}, []
    for name, fn in policy_fns.items():
        env, traj, soc = record_episode(fn, env_cfg, seed=SEED)
        records[name] = (env, traj, soc)
        rows.append({**summarize_traj(env, traj, environment), "policy": name})
    plot_trajectories_3d(records, f"trajectories_{tag}",
                         f"({ENV_LABELS.get(environment, tag)})")
    return records, rows


TRAJ_COLS = ["policy", "environment", "scenario", "obstacles_enabled", "n_obstacles",
             "tasks_completed", "tasks_total", "coverage_pct",
             "n_stranded", "mean_final_soc", "steps_taken", "blocked_moves"]


def write_traj_csv(rows, csv_path):
    write_csv(rows, csv_path, fieldnames=TRAJ_COLS)


def save_traj_exports(rows, tag, environment):
    json_path = os.path.join(OUT_DIR, f"trajectories_{tag}.json")
    csv_path = os.path.join(OUT_DIR, f"trajectories_{tag}.csv")
    write_json({"config": CFG_NAME, "environment": environment, "seed": SEED,
                "label": ENV_LABELS.get(environment, tag),
                "policies": {r["policy"]: r for r in rows}}, json_path)
    write_traj_csv(rows, csv_path)
    sub = os.path.join(OUT_DIR, "by_environment", environment)
    write_json({"config": CFG_NAME, "environment": environment, "seed": SEED,
                "policies": {r["policy"]: r for r in rows}},
               os.path.join(sub, "trajectories.json"))
    write_traj_csv(rows, os.path.join(sub, "trajectories.csv"))
    print(f"saved outputs/trajectories_{tag}.json + .csv")


all_traj_rows = []
traj_by_env = {}
for eid in ENV_ORDER:
    print(f"trajectories @ {eid} ...", flush=True)
    recs, rows = record_and_plot(POLICIES, ENV_CFGS[eid], eid, eid)
    save_traj_exports(rows, eid, eid)
    all_traj_rows.extend(rows)
    traj_by_env[eid] = {r["policy"]: r for r in rows}

write_traj_csv(all_traj_rows, os.path.join(OUT_DIR, "trajectories.csv"))
write_json({"config": CFG_NAME, "seed": SEED, "environments": ENV_ORDER,
            "n_obstacles": NFZ_N_OBSTACLES, "obstacle_half": NFZ_OBSTACLE_HALF,
            "conditions": traj_by_env},
           os.path.join(OUT_DIR, "trajectories.json"))
print("saved outputs/trajectories.csv + trajectories.json (all 4 environments)")


## 8.7 Proposed model: HRL-H and its ablations (all 4 environments)

One **2x2 figure per environment** (open, layered, open+NFZ, layered+NFZ) comparing
HRL-H vs HRL-H-NAV vs HRL-H-QMIX at the base config. Values also go to JSON/CSV.


In [ ]:
ABLATION_KEYS = ["hrlh", "hrlh_nav", "hrlh_qmix"]
ABLATION_LABELS = {"hrlh": "HRL-H (MAPPO)",
                   "hrlh_nav": "HRL-H-NAV (no heuristic)",
                   "hrlh_qmix": "HRL-H-QMIX (QMIX high level)"}
ABLATION_COLORS = {"hrlh": POLICY_COLORS["hrlh"], "hrlh_nav": "#8c564b",
                   "hrlh_qmix": "#2ca02c"}
ABLATION_MARKERS = {"hrlh": "h", "hrlh_nav": "v", "hrlh_qmix": "<"}


def plot_ablation_one_env(ablation_results, fname, title_tag):
    """One environment: 2x2 bars for the three HRL-H variants."""
    keys = ABLATION_KEYS
    metrics = [("coverage_mean", "Coverage (%)", "coverage_std"),
               ("stranded_mean", "Stranded UAVs (%)", "stranded_std"),
               ("energy_wh_per_task_mean", "Energy per task (Wh)",
                "energy_wh_per_task_std"),
               ("steps_mean", "Episode length (steps)", "steps_std")]
    fig, axes = plt.subplots(2, 2, figsize=figsize_for(2, 2))
    x = np.arange(len(keys)); width = 0.55
    for ax, (mk, label, sk) in zip(axes.ravel(), metrics):
        vals = [ablation_results[n][mk] for n in keys]
        errs = [ablation_results[n][sk] for n in keys]
        ax.bar(x, vals, width, yerr=errs, capsize=5,
               color=[ABLATION_COLORS[n] for n in keys],
               edgecolor="k", linewidth=0.5)
        for b, v in zip(ax.patches, vals):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}",
                    ha="center", va="bottom", fontsize=FONT_ANNOT)
        ax.set_xticks(x)
        ax.set_xticklabels([ABLATION_LABELS[n] for n in keys],
                           fontsize=FONT_TICK, rotation=12, ha="right")
        ax.set_ylabel(label, fontsize=FONT_LABEL)
        ax.set_title(label, fontsize=FONT_TITLE)
        ax.tick_params(labelsize=FONT_TICK)
    fig.suptitle(f"HRL-H ablation -- {title_tag} ({CFG_NAME})",
                 fontsize=FONT_SUPTITLE)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    save_fig(fig, fname)


ablation_by_env = {}
for eid, cfg in ENV_CFGS.items():
    print(f"HRL-H ablations @ {eid} ...", flush=True)
    ablation_runs = {}
    for name in ABLATION_KEYS:
        fn = ABLATIONS[name] if name != "hrlh" else hrlh
        ablation_runs[name] = [run_episode(fn, cfg, s) for s in range(N_SEEDS)]
    ablation_by_env[eid] = {name: summarize_runs(runs, 0)
                            for name, runs in ablation_runs.items()}
    plot_ablation_one_env(ablation_by_env[eid],
                          f"hrlh_ablation_comparison_{eid}",
                          ENV_LABELS[eid])

dump_base_by_env(ablation_by_env, OUT_DIR, "eval_hrlh",
                 {"config": CFG_NAME, "n_seeds": N_SEEDS,
                  "variants": ABLATION_KEYS, "labels": ABLATION_LABELS})
ablation_results = ablation_by_env["layered"]
print("saved outputs/eval_hrlh.json + .csv and 4 ablation figures")


## 8.8 Research extension A: time-critical tasks (all 4 environments)

Tasks expire after `task_deadline_steps` and reward decays per step. Sweep
(no deadline / lenient / tight) is saved per environment as JSON + CSV (tables only).


In [ ]:
DEADLINE_LEVELS = [0, 200, 100]     # steps; 0 = no deadline (static baseline)
DEADLINE_LABELS = {0: "none", 200: "lenient (200)", 100: "tight (100)"}
VALUE_DECAY = 0.005                  # per-step value decay at all deadline levels


def run_deadline_sweep(policy_fns, env_cfg, levels, n_seeds):
    results = {}
    for name, fn in policy_fns.items():
        results[name] = []
        for d in levels:
            cfg = EnvConfig3D(**{**env_cfg.__dict__, "task_mode": "time_critical",
                                 "task_deadline_steps": d,
                                 "task_value_decay_per_step": VALUE_DECAY if d > 0 else 0.0})
            runs = [run_episode(fn, cfg, s) for s in range(n_seeds)]
            results[name].append(summarize_runs(runs, d))
    return results


deadline_results_by_env = {}
for eid, cfg in ENV_CFGS.items():
    print(f"deadline sweep @ {eid} ...", flush=True)
    deadline_results_by_env[eid] = run_deadline_sweep(
        POLICIES, cfg, DEADLINE_LEVELS, N_SEEDS)
dump_sweep_by_env(deadline_results_by_env, OUT_DIR, "eval_deadline",
                  {"config": CFG_NAME, "n_seeds": N_SEEDS, "levels": DEADLINE_LEVELS,
                   "labels": DEADLINE_LABELS, "value_decay": VALUE_DECAY})
deadline_results = deadline_results_by_env["layered"]
print("saved outputs/eval_deadline.json + .csv (per environment)")


## 8.9 Research extension B: coupled (multi-agent) tasks (all 4 environments)

A task with requirement R[k] needs R[k] distinct agents. Sweep
(all-single / half need-2 / all need-2) is saved per environment as JSON + CSV (tables only).


In [ ]:
COUPLED_PROFILES = [
    ("all single-agent", 1, ()),
    ("half need 2", 1.5, tuple(2 if k % 2 == 0 else 1
                               for k in range(ENV_CFG.n_tasks))),
    ("all need 2", 2.0, tuple(2 for _ in range(ENV_CFG.n_tasks))),
]


def run_coupled_sweep(policy_fns, env_cfg, profiles, n_seeds):
    results = {}
    for name, fn in policy_fns.items():
        results[name] = []
        for label, value, req in profiles:
            cfg = EnvConfig3D(**{**env_cfg.__dict__, "task_requirements": req})
            runs = [run_episode(fn, cfg, s) for s in range(n_seeds)]
            results[name].append(summarize_runs(runs, value))
    return results


coupled_results_by_env = {}
for eid, cfg in ENV_CFGS.items():
    print(f"coupled sweep @ {eid} ...", flush=True)
    coupled_results_by_env[eid] = run_coupled_sweep(
        POLICIES, cfg, COUPLED_PROFILES, N_SEEDS)
dump_sweep_by_env(coupled_results_by_env, OUT_DIR, "eval_coupled",
                  {"config": CFG_NAME, "n_seeds": N_SEEDS,
                   "profiles": [(p[0], p[1], list(p[2])) for p in COUPLED_PROFILES]})
coupled_results = coupled_results_by_env["layered"]
print("saved outputs/eval_coupled.json + .csv (per environment)")


## 8.10 Research extension C: obstacle-density sweep (open + layered)

Varies obstacle count (0 / 2 / 4) on open and layered layouts. Saved as JSON + CSV
(tables only; the first-class NFZ environments are `open_nfz` / `layered_nfz`).


In [ ]:
OBSTACLE_LEVELS = [0, 2, 4]
OBSTACLE_HALF = NFZ_OBSTACLE_HALF


def run_obstacle_sweep(policy_fns, env_cfg, levels, n_seeds):
    results = {}
    for name, fn in policy_fns.items():
        results[name] = []
        for nb in levels:
            cfg = EnvConfig3D(**{**env_cfg.__dict__,
                                 "obstacles_enabled": nb > 0, "n_obstacles": nb,
                                 "obstacle_half": OBSTACLE_HALF})
            runs = [run_episode(fn, cfg, s) for s in range(n_seeds)]
            results[name].append(summarize_runs(runs, nb))
    return results


obstacle_results_by_env = {}
for eid in ["open", "layered"]:
    print(f"obstacle-density sweep @ {eid} ...", flush=True)
    obstacle_results_by_env[eid] = run_obstacle_sweep(
        POLICIES, ENV_CFGS[eid], OBSTACLE_LEVELS, N_SEEDS)
dump_sweep_by_env(obstacle_results_by_env, OUT_DIR, "eval_obstacles",
                  {"config": CFG_NAME, "n_seeds": N_SEEDS, "levels": OBSTACLE_LEVELS,
                   "obstacle_half": OBSTACLE_HALF})
obstacle_results = obstacle_results_by_env["layered"]
print("saved outputs/eval_obstacles.json + .csv")


## 9. Summary figures, extended metrics, and comparison tables

Per environment (4 figures each): base-config comparison, extended metrics, and
(from section 8.5 / 8.7) trajectories and HRL-H ablations. Scenario comparison
is one figure with all 4 environments. Sweep results are tables only.


In [ ]:
def plot_summary_bars(base_results, fname, title_tag=""):
    metrics = [("coverage_mean", "Coverage (%)", "coverage_std"),
               ("stranded_mean", "Stranded UAVs (%)", "stranded_std"),
               ("reward_mean", "Total reward", "reward_std"),
               ("soc_mean", "Mean final SOC", "soc_std"),
               ("steps_mean", "Episode length (steps)", "steps_std")]
    nrows, ncols = panel_shape(len(metrics))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize_for(nrows, ncols, cell=(7.5, 5.6)))
    x = np.arange(len(POLICY_ORDER)); width = 0.68
    axes_flat = np.atleast_1d(axes).ravel()
    for ax, (mk, label, sk) in zip(axes_flat, metrics):
        vals = [base_results[name][mk] for name in POLICY_ORDER]
        errs = [base_results[name][sk] for name in POLICY_ORDER]
        bars = ax.bar(x, vals, width, yerr=errs, capsize=5,
                      color=[POLICY_COLORS[n] for n in POLICY_ORDER],
                      edgecolor="k", linewidth=0.5)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}",
                    ha="center", va="bottom", fontsize=FONT_ANNOT)
        ax.set_xticks(x)
        ax.set_xticklabels([POLICY_LABELS[n] for n in POLICY_ORDER],
                           fontsize=FONT_TICK - 1, rotation=22, ha="right")
        ax.set_ylabel(label, fontsize=FONT_LABEL)
        ax.set_title(label, fontsize=FONT_TITLE)
        ax.tick_params(labelsize=FONT_TICK)
    hide_unused_axes(axes_flat, len(metrics))
    fig.suptitle(f"3D base-config comparison -- all 8 approaches ({CFG_NAME})"
                 + (f" -- {title_tag}" if title_tag else ""),
                 fontsize=FONT_SUPTITLE)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    save_fig(fig, fname)


def plot_extended_metrics(base_results, fname, title_tag=""):
    metrics = [
        ("energy_wh_per_task_mean", "Energy per task (Wh)", "energy_wh_per_task_std"),
        ("dist_per_task_m_mean", "Distance per task (m)", "dist_per_task_m_std"),
        ("min_soc_mean", "Min SOC reached", "min_soc_std"),
        ("below_floor_pct_mean", "Time below safe floor (%)", "below_floor_pct_std"),
        ("contention_pct_mean", "Assignment contention (% steps)", "contention_pct_std"),
        ("info_age_mean", "Mean info age (steps)", "info_age_std"),
    ]
    nrows, ncols = panel_shape(len(metrics))  # 6 -> 3x2
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize_for(nrows, ncols))
    x = np.arange(len(POLICY_ORDER)); width = 0.68
    axes_flat = np.atleast_1d(axes).ravel()
    for ax, (mk, label, sk) in zip(axes_flat, metrics):
        vals = [base_results[name][mk] for name in POLICY_ORDER]
        errs = [base_results[name][sk] for name in POLICY_ORDER]
        bars = ax.bar(x, vals, width, yerr=errs, capsize=5,
                      color=[POLICY_COLORS[n] for n in POLICY_ORDER],
                      edgecolor="k", linewidth=0.5)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}",
                    ha="center", va="bottom", fontsize=FONT_ANNOT)
        ax.set_xticks(x)
        ax.set_xticklabels([POLICY_LABELS[n] for n in POLICY_ORDER],
                           fontsize=FONT_TICK - 1, rotation=22, ha="right")
        ax.set_ylabel(label, fontsize=FONT_LABEL)
        ax.set_title(label, fontsize=FONT_TITLE)
        ax.tick_params(labelsize=FONT_TICK)
    hide_unused_axes(axes_flat, len(metrics))
    fig.suptitle(f"3D extended operational metrics -- all 8 approaches ({CFG_NAME})"
                 + (f" -- {title_tag}" if title_tag else ""),
                 fontsize=FONT_SUPTITLE)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    save_fig(fig, fname)


def plot_latency_hist(latency_by_policy, fname, title_tag=""):
    fig, ax = plt.subplots(figsize=(14, 6))
    for name in POLICY_ORDER:
        lat = latency_by_policy.get(name) or []
        if not lat:
            continue
        ax.hist(lat, bins=25, alpha=0.5, color=POLICY_COLORS[name],
                label=f"{POLICY_LABELS[name]} (median {np.median(lat):.0f})")
    ax.set_xlabel("Task completion step", fontsize=FONT_LABEL)
    ax.set_ylabel("Tasks completed", fontsize=FONT_LABEL)
    ax.set_title(f"Per-task completion latency -- all 8 approaches ({CFG_NAME})"
                 + (f" -- {title_tag}" if title_tag else ""),
                 fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    ax.legend(loc="best", fontsize=FONT_LEGEND, title="Policy",
              title_fontsize=FONT_LEGEND, frameon=True)
    fig.tight_layout()
    save_fig(fig, fname)


# Reuse the base-config runs already collected in section 8.
base_results_by_env = scenario_results
base_runs_by_env = scenario_runs
latency_by_env = {
    eid: {name: sorted(s for r in runs for s in r["completion_steps"])
          for name, runs in runs_map.items()}
    for eid, runs_map in base_runs_by_env.items()
}

for eid in ENV_ORDER:
    tag = ENV_LABELS[eid]
    plot_summary_bars(base_results_by_env[eid], f"comparison_summary_{eid}", tag)
    plot_extended_metrics(base_results_by_env[eid], f"extended_metrics_{eid}", tag)
    plot_latency_hist(latency_by_env[eid], f"latency_histogram_{eid}", tag)

dump_base_by_env(base_results_by_env, OUT_DIR, "eval_base",
                 {"config": CFG_NAME, "n_seeds": N_SEEDS,
                  "n_agents": ENV_CFG.n_agents, "n_tasks": ENV_CFG.n_tasks})

# Comparison tables (CSV / JSON / Markdown) for later reuse
dump_tables(base_results_by_env, OUT_DIR, POLICY_ORDER, POLICY_LABELS,
            title=f"Base-config comparison ({CFG_NAME}, 8 UAVs / 8 tasks)")

base_results = base_results_by_env["layered"]
latency_by_policy = latency_by_env["layered"]

summary = {
    "project": "uav_swarm_3d",
    "environment": "3D",
    "config": CFG_NAME,
    "environments": ENV_ORDER,
    "env_labels": ENV_LABELS,
    "n_agents": ENV_CFG.n_agents,
    "n_seeds": N_SEEDS,
    "mappo_training_episodes": trained_episodes,
    "mappo_train_task_counts": TRAIN_TASK_COUNTS,
    "maddpg_training": maddpg_log,
    "qmix_training": maddpg_log,
    "policies": [POLICY_LABELS[n] for n in POLICY_ORDER],
    "base_config_results": base_results_by_env,
    "base_latency_stats": {
        eid: {name: {"median_steps": float(np.median(latency_by_env[eid][name]))
                     if latency_by_env[eid][name] else None,
                     "n_tasks": len(latency_by_env[eid][name])}
              for name in POLICY_ORDER}
        for eid in ENV_ORDER
    },
    "hrlh_ablation_results": ablation_by_env,
    "sweeps": {
        "packet_loss": {"levels": PL_LEVELS, "results": pl_results_by_env},
        "comm_range": {"levels": CR_LEVELS, "results": cr_results_by_env},
        "task_density": {"levels": TD_LEVELS, "results": td_results_by_env},
        "fault_tolerance": {"levels": FT_LEVELS, "results": ft_results_by_env},
        "scalability": {"levels": SC_LEVELS, "results": sc_results_by_env},
        "scenarios": {"environments": ENV_ORDER, "results": scenario_results},
    },
    "extensions": {
        "deadline": {"levels": DEADLINE_LEVELS, "results": deadline_results_by_env},
        "coupled": {"profiles": [(p[0], p[1], list(p[2])) for p in COUPLED_PROFILES],
                    "results": coupled_results_by_env},
        "obstacles": {"levels": OBSTACLE_LEVELS, "results": obstacle_results_by_env},
    },
}
write_json(summary, os.path.join(OUT_DIR, "summary.json"))
print("saved outputs/summary.json + comparison tables")


## 9.5 Discussion: honest findings

<RESULTS_PLACEHOLDER>

## 10. Outputs produced by this notebook

Figures (600 dpi PNG) live in `figures/`. Values live in `outputs/` as JSON **and** CSV.
Per-environment copies are under `outputs/by_environment/<open|layered|open_nfz|layered_nfz>/`.
Environment snapshots (for re-plotting the 2x2 figure later) are in `outputs/env_snapshots/`.
Comparison tables: `outputs/comparison_table.csv`, `.json`, `.md`.


In [ ]:
print("=== figures/ ===")
for f in sorted(os.listdir(FIG_DIR)):
    print(" ", f)
print("=== outputs/ ===")
for root, dirs, files in os.walk(OUT_DIR):
    rel = os.path.relpath(root, OUT_DIR)
    for f in sorted(files):
        print(" ", os.path.join(rel, f) if rel != "." else f)
print("\nAll figures: 600 dpi PNG. All values: JSON + CSV in outputs/ "
      "(per environment under outputs/by_environment/).")
